In [0]:
%sql
-- VERIFY CDC IS ENABLED
DESCRIBE HISTORY retail_lakehouse.silver.sales;


In [0]:
%sql
-- DAY 2 - SIMULATE INCREMENTAL INSERT
INSERT INTO retail_lakehouse.silver.sales
VALUES
(
    9999,      -- TransactionID
    1,         -- CustomerID
    1001,      -- ProductID
    100,       -- StoreID
    2,         -- Quantity
    CURRENT_DATE()
);


In [0]:
%sql
-- DAY 2 - SIMULATE DELETE
DELETE FROM retail_lakehouse.silver.sales
WHERE TransactionID = 9999;


In [0]:
%sql
-- CDC start version from DESCRIBE HISTORY
SELECT *
FROM table_changes(
    'retail_lakehouse.silver.sales',
    1
);


In [0]:
%sql
-- FACT TABLE INCREMENTAL MERGE
MERGE INTO retail_lakehouse.gold.fact_sales tgt
USING (
    SELECT
        s.TransactionID,
        c.CustomerSK,
        p.ProductSK,
        st.StoreSK,
        s.Quantity,
        s.Quantity * p.UnitPrice AS Amount,
        s.TxnDate
    FROM table_changes(
        'retail_lakehouse.silver.sales',
        1
    ) s

    JOIN retail_lakehouse.gold.dim_customer c
        ON s.CustomerID = c.CustomerID
        AND c.IsActive = TRUE

    JOIN retail_lakehouse.gold.dim_product p
        ON s.ProductID = p.ProductID

    JOIN retail_lakehouse.gold.dim_store st
        ON s.StoreID = st.StoreID

    WHERE s._change_type = 'insert'

) src

ON tgt.TransactionID = src.TransactionID
WHEN NOT MATCHED THEN
INSERT
(
    SalesSK,
    TransactionID,
    CustomerSK,
    ProductSK,
    StoreSK,
    Quantity,
    Amount,
    TxnDate
)
VALUES
(
    monotonically_increasing_id(),
    src.TransactionID,
    src.CustomerSK,
    src.ProductSK,
    src.StoreSK,
    src.Quantity,
    src.Amount,
    src.TxnDate
);


In [0]:
%sql
-- VERIFY FACT TABLE AFTER INCREMENTAL LOAD
SELECT *
FROM retail_lakehouse.gold.fact_sales
ORDER BY TransactionID DESC;

In [0]:
%sql
-- FULL LOAD vs INCREMENTAL LOAD VALIDATION
SELECT COUNT(*) AS fact_sales_count
FROM retail_lakehouse.gold.fact_sales;


In [0]:
%sql
-- DUPLICATE VALIDATION
SELECT
    TransactionID,
    COUNT(*)
FROM retail_lakehouse.gold.fact_sales
GROUP BY TransactionID
HAVING COUNT(*) > 1;

In [0]:
%sql
-- CDC VALIDATION
SELECT
    _change_type,
    _commit_version,
    _commit_timestamp,
    TransactionID
FROM table_changes(
    'retail_lakehouse.silver.sales',
    1
)
ORDER BY _commit_version DESC;

In [0]:
%sql
-- INCREMENTAL LOAD VALIDATION
-- Expected:
-- Only changed/new rows should process
-- No full reload should happen

SELECT COUNT(*)
FROM retail_lakehouse.gold.fact_sales;